In [11]:
import sys
sys.path.insert(0, "/Users/nautilus/cobblestone_study")

import pandas as pd
from config import CLEAN_PARQUET, PEAK_HOURS, PEAK_WEEKDAYS
from src.features import build_features, FEATURE_COLS
from src.models import train_lgbm, predict_lgbm

df = pd.read_parquet("/Users/nautilus/cobblestone_study/data/clean.parquet")
df = build_features(df)
df.shape


(17181, 17)

In [12]:
prices = df["prices"]

# Baseload = avg of All hours
baseload_mean = prices.mean()

# Peak = avg of peak hours
is_peak = prices.index.hour.isin(PEAK_HOURS) & prices.index.dayofweek.isin(PEAK_WEEKDAYS)
peak_mean = prices[is_peak].mean()

print(baseload_mean, peak_mean)

91.06301423083639 95.6911515299479


In [13]:
# Forecast
PROMPT_HOURS = 7 * 24 # Front week
forecast_df = df.iloc[-PROMPT_HOURS:]   # last 7 days
train_df = df.iloc[:-PROMPT_HOURS]  # Everything before

model = train_lgbm(train_df[FEATURE_COLS], train_df["prices"])
forecast = predict_lgbm(model, forecast_df[FEATURE_COLS])

print(f"Forecasted {len(forecast)} hours")
print(f"Forecast period: {forecast.index[0]} → {forecast.index[-1]}")


Forecasted 168 hours
Forecast period: 2026-06-05 00:00:00+02:00 → 2026-06-11 23:00:00+02:00


In [14]:
# Forecast to baseload
forecast_base_mean = forecast.mean()
forecast_is_peak = forecast.index.hour.isin(PEAK_HOURS) & forecast.index.dayofweek.isin(PEAK_WEEKDAYS)
forecast_peak_mean = forecast[forecast_is_peak].mean()

# Reference (avg 4 weeks)
REFERENCE_WEEKS = 4
reference_window = train_df['prices'].iloc[-(REFERENCE_WEEKS * PROMPT_HOURS):]
reference_base_mean = reference_window.mean()

reference_is_peak = reference_window.index.hour.isin(PEAK_HOURS) & reference_window.index.dayofweek.isin(PEAK_WEEKDAYS)
reference_peak_mean = reference_window[reference_is_peak].mean()

print(f"Baseload — forecast: {forecast_base_mean:.2f}  reference: {reference_base_mean:.2f}  spread: {forecast_base_mean - reference_base_mean:+.2f}")
print(f"Peak     — forecast: {forecast_peak_mean:.2f}  reference: {reference_peak_mean:.2f}  spread: {forecast_peak_mean - reference_peak_mean:+.2f}")

Baseload — forecast: 93.55  reference: 100.41  spread: -6.86
Peak     — forecast: 76.01  reference: 78.59  spread: -2.57


In [20]:
import json
with open("/Users/nautilus/cobblestone_study/outputs/metrics.json") as f:
    metrics = json.load(f)
    
mae = metrics["lgbm"]["mae"]
rmse = metrics["lgbm"]["rmse"]

def signal_to_noise(spread, mae):
    ratio = abs(spread) / mae
    if ratio < 1.0:
        return "LOW"      # spread is inside our own error → noise
    elif ratio < 2.0:
        return "MEDIUM"
    else:
        return "HIGH"

def zscore(spread, rmse):
    return spread / rmse


print(f"Model MAE: {mae}")
print(f"Baseload spread -6.86 → {signal_to_noise(-6.86, mae)}")
print(f"Peak spread -2.57 → {signal_to_noise(-2.57, mae)}")

print(f"Model RMSE: {rmse}")
print(f"Baseload spread -6.86 → {zscore(-6.86, rmse):.2f}")
print(f"Peak spread -2.57 → {zscore(-2.57, rmse):.2f}")


Model MAE: 16.66
Baseload spread -6.86 → LOW
Peak spread -2.57 → LOW
Model RMSE: 28.58
Baseload spread -6.86 → -0.24
Peak spread -2.57 → -0.09
